[Reference](https://medium.com/@pankaj_pandey/09633a66cf5a)

In [1]:
from openai import OpenAI

client = OpenAI()
response = client.chat.completions.create(
    model="gpt-5.1",
    messages=[{"role": "user", "content": "Summarize our Q4 results"}],
)
print(response.choices[0].message.content)

# Step 1: Build the agent

In [2]:
from agents import Agent

triage = Agent(
    name="Router",
    instructions="Route simple questions to Fast, complex ones to Deep.",
    handoffs=["fast_agent", "deep_agent"],
)
fast_agent = Agent(
    name="Fast",
    model="gpt-4.1-mini",
    instructions="Answer quick factual questions.",
)
deep_agent = Agent(
    name="Deep",
    model="gpt-5.2",
    instructions="Handle complex analysis and reasoning.",
)

In [3]:
# Agno (from official GitHub README)
from agno.agent import Agent
from agno.models.anthropic import Claude
from agno.tools.mcp import MCPTools

agent = Agent(
    model=Claude(id="claude-sonnet-4-5"),
    tools=[MCPTools(url="https://docs.agno.com/mcp")],
)


In [4]:
from mem0 import Memory

m = Memory()
m.add("Prefers Postgres over MySQL, cost-sensitive on infra", user_id="alice")
results = m.search("database preferences", user_id="alice")

In [5]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.db.postgres import PostgresDb

agent = Agent(
    model=OpenAIChat(id="gpt-4.1"),
    db=PostgresDb(db_url="postgresql+psycopg://ai:ai@localhost:5532/ai"),
    enable_agentic_memory=True,
)

# Step 2: Serve it as an API


In [6]:
from agno.agent import Agent
from agno.db.sqlite import SqliteDb
from agno.models.anthropic import Claude
from agno.os import AgentOS
from agno.tools.mcp import MCPTools

agent = Agent(
    name="Agno Assist",
    model=Claude(id="claude-sonnet-4-5"),
    db=SqliteDb(db_file="agno.db"),
    tools=[MCPTools(url="https://docs.agno.com/mcp")],
    add_history_to_context=True,
    num_history_runs=3,
)
agent_os = AgentOS(agents=[agent], tracing=True)
app = agent_os.get_app()
# Run: fastapi dev agno_assist.py

In [7]:
from langgraph_sdk import get_sync_client

client = get_sync_client(
    url="https://my-deployment.langsmith.dev",
    api_key="ls-..."
)
for chunk in client.runs.stream(
    None,
    "agent",
    input={"messages": [{"role": "human", "content": "Summarize Q4 results"}]},
    stream_mode="updates",
):
    print(chunk.data)

# Step 3: Connect it to users

In [8]:
from agno.agent import Agent
from agno.models.openai import OpenAIResponses
from agno.os import AgentOS
from agno.os.interfaces.agui import AGUI


chat_agent = Agent(model=OpenAIResponses(id="gpt-5.2"))
agent_os = AgentOS(
    agents=[chat_agent],
    interfaces=[AGUI(agent=chat_agent)],
)
app = agent_os.get_app()